# test_jointly_ntv

> 对应代码：`EconometricsCode/code_in_notes/Chap.9/test_jointly_ntv.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.9`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.9")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

加载俄罗斯国家电视台（NTV）报道与选举数据，并声明面板结构：个体为 tik_id（选区），时间为 year。

In [ ]:
%%stata
clear
set more off
use ../datasets/NTV_Aggregate_Data_reshaped.dta
xtset tik_id year

先做不加控制的基准检验：用 reghdfe 把 NTV 报道量对主要政党的滞后四期得票率（L4.）以及投票率做回归，absorb(region) 吸收地区固定效应。随后的 test 命令联合检验所有政党得票率与投票率的系数是否联合为零——这是一个典型的 Wald 联合显著性检验，原假设下 F 统计量服从 F 分布。

In [ ]:
%%stata
// 不进行控制进行检验
reghdfe NTV L4.Votes_NDR_ L4.Votes_SPS_ L4.Votes_Yabloko_ L4.Votes_KPRF_ L4.Votes_LDPR_ L4.Turnout if year==1999, absorb(region) 
test L4.Votes_NDR_ L4.Votes_SPS_ L4.Votes_Yabloko_ L4.Votes_KPRF_ L4.Votes_LDPR_ L4.Turnout

生成对数人口与对数工资的 1–5 阶多项式作为灵活的控制变量，用以捕捉社会经济条件的非线性影响。

In [ ]:
%%stata
// 控制社会经济变量以及人口、工资的多项式
forvalues i=1/5{
    gen log_pop_`i'=logpop^`i'
    gen log_wage_`i'=log_wage^`i'
}

把滞后一期的社会经济控制变量（人口与工资多项式、护士数、人均医生数）存入局部宏 E_controls，便于书写回归式。

In [ ]:
%%stata
local E_controls "L.log_pop_* L.log_wage_* L.nurses L.doctors_pc"

加入控制变量后重新回归，并用 cluster(region) 计算地区层面的聚类稳健标准误，以允许同一地区内误差相关。随后再次做联合显著性检验：比较两组结果可以看到，控制可观测特征后选举变量对 NTV 报道的联合解释力是否仍然存在，从而判断媒体报道是否被政治因素驱动。

In [ ]:
%%stata
reghdfe NTV L4.Votes_NDR_ L4.Votes_SPS_ L4.Votes_Yabloko_ L4.Votes_KPRF_ L4.Votes_LDPR_ L4.Turnout `E_controls' if year==1999, absorb(region) cluster(region)
test L4.Votes_NDR_ L4.Votes_SPS_ L4.Votes_Yabloko_ L4.Votes_KPRF_ L4.Votes_LDPR_ L4.Turnout